In [2]:
"""
SCRIPT DE PRÉTRAITEMENT COMPLET POUR LE DATASET DE RECYCLAGE
==============================================================

Ce script implémente TOUS les prétraitements possibles avec une logique
conditionnelle basée sur les relations entre colonnes.

Auteur: Claude
Date: 2026
"""

import pandas as pd
import numpy as np
import re
from scipy import stats
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

class PreprocesseurRecyclage:
    """Classe pour prétraiter le dataset de recyclage avec logique avancée"""
    
    def __init__(self, df):
        self.df = df.copy()
        self.df_original = df.copy()
        self.rapport_modifications = []
        
    def ajouter_rapport(self, etape, details):
        """Ajoute une entrée au rapport de modifications"""
        self.rapport_modifications.append({
            'Étape': etape,
            'Détails': details
        })
    
    def extraire_infos_rapport(self):
        """
        ÉTAPE 1: EXTRACTION D'INFORMATIONS DU RAPPORT_COLLECTE
        ========================================================
        Extrait les informations numériques cachées dans le texte du rapport
        pour combler certaines valeurs manquantes avec logique.
        """
        print("\n" + "="*80)
        print("ÉTAPE 1: EXTRACTION D'INFORMATIONS DU RAPPORT_COLLECTE")
        print("="*80)
        
        infos_extraites = {
            'poids_extrait': [],
            'volume_extrait': [],
            'rigidite_extrait': []
        }
        
        for idx, rapport in enumerate(self.df['Rapport_Collecte']):
            # Extraire le poids (kg)
            poids_match = re.search(r'(\d+\.?\d*)\s*kg', rapport, re.IGNORECASE)
            if poids_match:
                infos_extraites['poids_extrait'].append(float(poids_match.group(1)))
            else:
                infos_extraites['poids_extrait'].append(np.nan)
            
            # Extraire le volume (L)
            volume_match = re.search(r'(\d+\.?\d*)\s*L', rapport, re.IGNORECASE)
            if volume_match:
                infos_extraites['volume_extrait'].append(float(volume_match.group(1)))
            else:
                infos_extraites['volume_extrait'].append(np.nan)
            
            # Extraire la rigidité qualitative
            if 'extrêmement rigide' in rapport.lower() or 'très rigide' in rapport.lower():
                infos_extraites['rigidite_extrait'].append(10)
            elif 'rigide' in rapport.lower():
                infos_extraites['rigidite_extrait'].append(7)
            elif 'semi-rigide' in rapport.lower():
                infos_extraites['rigidite_extrait'].append(5)
            elif 'souple' in rapport.lower():
                infos_extraites['rigidite_extrait'].append(3)
            else:
                infos_extraites['rigidite_extrait'].append(np.nan)
        
        # Ajouter les colonnes extraites
        self.df['Poids_Extrait'] = infos_extraites['poids_extrait']
        self.df['Volume_Extrait'] = infos_extraites['volume_extrait']
        self.df['Rigidite_Extrait'] = infos_extraites['rigidite_extrait']
        
        print(f"✓ Poids extraits du rapport: {pd.notna(self.df['Poids_Extrait']).sum()}")
        print(f"✓ Volumes extraits du rapport: {pd.notna(self.df['Volume_Extrait']).sum()}")
        print(f"✓ Rigidités extraites du rapport: {pd.notna(self.df['Rigidite_Extrait']).sum()}")
        
        self.ajouter_rapport('Extraction infos rapport', 
                            f"Extraits: {pd.notna(self.df['Poids_Extrait']).sum()} poids, "
                            f"{pd.notna(self.df['Volume_Extrait']).sum()} volumes, "
                            f"{pd.notna(self.df['Rigidite_Extrait']).sum()} rigidités")
    
    def traiter_doublons(self):
        """
        ÉTAPE 2: TRAITEMENT DES DOUBLONS
        ==================================
        Stratégie intelligente basée sur la qualité des données
        """
        print("\n" + "="*80)
        print("ÉTAPE 2: TRAITEMENT DES DOUBLONS")
        print("="*80)
        
        nb_doublons_initial = self.df.duplicated().sum()
        print(f"Nombre de doublons détectés: {nb_doublons_initial}")
        
        if nb_doublons_initial > 0:
            # Créer un score de qualité pour chaque ligne
            self.df['Score_Qualite'] = 0
            
            # Plus de valeurs non-nulles = meilleure qualité
            for col in ['Poids', 'Volume', 'Conductivite', 'Opacite', 'Rigidite', 'Prix_Revente']:
                self.df['Score_Qualite'] += self.df[col].notna().astype(int)
            
            # Garder le doublon avec le meilleur score de qualité
            self.df = self.df.sort_values('Score_Qualite', ascending=False)
            self.df = self.df.drop_duplicates(keep='first')
            self.df = self.df.drop('Score_Qualite', axis=1)
            self.df = self.df.reset_index(drop=True)
            
            nb_doublons_final = nb_doublons_initial
            print(f"✓ {nb_doublons_final} doublons supprimés (gardé les lignes avec plus de données)")
            self.ajouter_rapport('Suppression doublons', f"{nb_doublons_final} lignes supprimées")
    
    def traiter_valeurs_manquantes_categorielles(self):
        """
        ÉTAPE 3: TRAITEMENT DES VALEURS MANQUANTES - COLONNES CATÉGORIELLES
        =====================================================================
        Utilise le rapport de collecte pour déduire la catégorie et la source
        """
        print("\n" + "="*80)
        print("ÉTAPE 3: TRAITEMENT VALEURS MANQUANTES - CATÉGORIELLES")
        print("="*80)
        
        # 3A. Catégorie manquante - déduction du rapport
        cat_missing_initial = self.df['Categorie'].isna().sum()
        
        for idx, row in self.df[self.df['Categorie'].isna()].iterrows():
            rapport = row['Rapport_Collecte'].lower()
            
            if 'papier' in rapport or 'carton' in rapport or 'feuilles' in rapport:
                self.df.at[idx, 'Categorie'] = 'Papier'
            elif 'plastique' in rapport or 'emballage' in rapport:
                self.df.at[idx, 'Categorie'] = 'Plastique'
            elif 'verre' in rapport or 'contenants' in rapport or 'bouteille' in rapport:
                self.df.at[idx, 'Categorie'] = 'Verre'
            elif 'métal' in rapport or 'ferraille' in rapport or 'aluminium' in rapport:
                self.df.at[idx, 'Categorie'] = 'Métal'
        
        cat_remplis = cat_missing_initial - self.df['Categorie'].isna().sum()
        print(f"✓ Catégorie: {cat_remplis}/{cat_missing_initial} valeurs remplies par déduction")
        
        # 3B. Source manquante - déduction du rapport
        source_missing_initial = self.df['Source'].isna().sum()
        
        for idx, row in self.df[self.df['Source'].isna()].iterrows():
            rapport = row['Rapport_Collecte'].lower()
            
            if 'usine a' in rapport or "l'usine a" in rapport:
                self.df.at[idx, 'Source'] = 'Usine_A'
            elif 'usine b' in rapport or "l'usine b" in rapport:
                self.df.at[idx, 'Source'] = 'Usine_B'
            elif 'centre de tri' in rapport or 'centre tri' in rapport:
                self.df.at[idx, 'Source'] = 'Centre_Tri'
            elif 'collecte citoyenne' in rapport or 'citoyenne' in rapport:
                self.df.at[idx, 'Source'] = 'Collecte_Citoyenne'
        
        source_remplis = source_missing_initial - self.df['Source'].isna().sum()
        print(f"✓ Source: {source_remplis}/{source_missing_initial} valeurs remplies par déduction")
        
        # 3C. Remplir les catégories/sources restantes par le mode selon l'autre variable
        if self.df['Categorie'].isna().sum() > 0:
            mode_cat = self.df['Categorie'].mode()[0]
            self.df['Categorie'].fillna(mode_cat, inplace=True)
            print(f"✓ Catégories restantes remplies avec mode: {mode_cat}")
        
        if self.df['Source'].isna().sum() > 0:
            mode_source = self.df['Source'].mode()[0]
            self.df['Source'].fillna(mode_source, inplace=True)
            print(f"✓ Sources restantes remplies avec mode: {mode_source}")
        
        self.ajouter_rapport('Valeurs manquantes catégorielles', 
                            f"Catégorie: {cat_remplis} déduites, Source: {source_remplis} déduites")
    
    def traiter_valeurs_manquantes_numeriques(self):
        """
        ÉTAPE 4: TRAITEMENT DES VALEURS MANQUANTES - COLONNES NUMÉRIQUES
        ==================================================================
        Stratégie par colonne avec logique conditionnelle
        """
        print("\n" + "="*80)
        print("ÉTAPE 4: TRAITEMENT VALEURS MANQUANTES - NUMÉRIQUES")
        print("="*80)
        
        # 4A. POIDS - Utiliser l'extraction du rapport ou calculer selon Volume/Catégorie
        poids_missing = self.df['Poids'].isna().sum()
        
        # D'abord, utiliser les poids extraits du rapport
        mask_poids_na = self.df['Poids'].isna()
        self.df.loc[mask_poids_na, 'Poids'] = self.df.loc[mask_poids_na, 'Poids_Extrait']
        remplis_extrait = poids_missing - self.df['Poids'].isna().sum()
        
        # Ensuite, estimer avec densité typique par catégorie si Volume disponible
        densites = {
            'Papier': 0.7,      # kg/L
            'Plastique': 0.95,  # kg/L
            'Verre': 2.5,       # kg/L
            'Métal': 7.0        # kg/L
        }
        
        for cat, densite in densites.items():
            mask = (self.df['Poids'].isna()) & (self.df['Categorie'] == cat) & (self.df['Volume'].notna())
            self.df.loc[mask, 'Poids'] = self.df.loc[mask, 'Volume'] * densite
        
        remplis_calcul = poids_missing - remplis_extrait - self.df['Poids'].isna().sum()
        
        # Pour le reste, médiane par catégorie
        for cat in self.df['Categorie'].unique():
            mask = (self.df['Poids'].isna()) & (self.df['Categorie'] == cat)
            mediane = self.df[self.df['Categorie'] == cat]['Poids'].median()
            if pd.notna(mediane):
                self.df.loc[mask, 'Poids'] = mediane
        
        remplis_mediane = poids_missing - remplis_extrait - remplis_calcul - self.df['Poids'].isna().sum()
        
        print(f"✓ Poids: {remplis_extrait} du rapport + {remplis_calcul} calculés (densité×volume) + "
              f"{remplis_mediane} par médiane")
        
        # 4B. VOLUME - Utiliser l'extraction du rapport ou calculer depuis Poids
        volume_missing = self.df['Volume'].isna().sum()
        
        # D'abord, utiliser les volumes extraits
        mask_vol_na = self.df['Volume'].isna()
        self.df.loc[mask_vol_na, 'Volume'] = self.df.loc[mask_vol_na, 'Volume_Extrait']
        vol_extrait = volume_missing - self.df['Volume'].isna().sum()
        
        # Calculer depuis le poids si disponible
        for cat, densite in densites.items():
            mask = (self.df['Volume'].isna()) & (self.df['Categorie'] == cat) & (self.df['Poids'].notna())
            self.df.loc[mask, 'Volume'] = self.df.loc[mask, 'Poids'] / densite
        
        vol_calcul = volume_missing - vol_extrait - self.df['Volume'].isna().sum()
        
        # Médiane par catégorie pour le reste
        for cat in self.df['Categorie'].unique():
            mask = (self.df['Volume'].isna()) & (self.df['Categorie'] == cat)
            mediane = self.df[self.df['Categorie'] == cat]['Volume'].median()
            if pd.notna(mediane):
                self.df.loc[mask, 'Volume'] = mediane
        
        vol_mediane = volume_missing - vol_extrait - vol_calcul - self.df['Volume'].isna().sum()
        
        print(f"✓ Volume: {vol_extrait} du rapport + {vol_calcul} calculés (poids/densité) + "
              f"{vol_mediane} par médiane")
        
        # 4C. CONDUCTIVITÉ - Dépend fortement de la catégorie
        cond_missing = self.df['Conductivite'].isna().sum()
        
        # Métal = haute conductivité, autres = 0
        self.df.loc[self.df['Conductivite'].isna() & (self.df['Categorie'] == 'Métal'), 'Conductivite'] = \
            self.df[self.df['Categorie'] == 'Métal']['Conductivite'].median()
        
        # Non-métaux = 0 (isolants)
        self.df.loc[self.df['Conductivite'].isna() & (self.df['Categorie'] != 'Métal'), 'Conductivite'] = 0.0
        
        print(f"✓ Conductivité: {cond_missing} remplies selon catégorie (Métal=médiane, autres=0)")
        
        # 4D. OPACITÉ - Par médiane selon catégorie
        opac_missing = self.df['Opacite'].isna().sum()
        
        for cat in self.df['Categorie'].unique():
            mask = (self.df['Opacite'].isna()) & (self.df['Categorie'] == cat)
            mediane = self.df[self.df['Categorie'] == cat]['Opacite'].median()
            if pd.notna(mediane):
                self.df.loc[mask, 'Opacite'] = mediane
        
        print(f"✓ Opacité: {opac_missing} remplies par médiane selon catégorie")
        
        # 4E. RIGIDITÉ - Utiliser l'extraction ou médiane par catégorie
        rig_missing = self.df['Rigidite'].isna().sum()
        
        # D'abord utiliser les rigidités extraites
        mask_rig_na = self.df['Rigidite'].isna()
        self.df.loc[mask_rig_na, 'Rigidite'] = self.df.loc[mask_rig_na, 'Rigidite_Extrait']
        rig_extrait = rig_missing - self.df['Rigidite'].isna().sum()
        
        # Puis médiane par catégorie
        for cat in self.df['Categorie'].unique():
            mask = (self.df['Rigidite'].isna()) & (self.df['Categorie'] == cat)
            mediane = self.df[self.df['Categorie'] == cat]['Rigidite'].median()
            if pd.notna(mediane):
                self.df.loc[mask, 'Rigidite'] = mediane
        
        rig_mediane = rig_missing - rig_extrait - self.df['Rigidite'].isna().sum()
        
        print(f"✓ Rigidité: {rig_extrait} du rapport + {rig_mediane} par médiane")
        
        # 4F. PRIX_REVENTE - Par médiane selon catégorie et source
        prix_missing = self.df['Prix_Revente'].isna().sum()
        
        for cat in self.df['Categorie'].unique():
            for source in self.df['Source'].unique():
                mask = (self.df['Prix_Revente'].isna()) & \
                       (self.df['Categorie'] == cat) & \
                       (self.df['Source'] == source)
                mediane = self.df[(self.df['Categorie'] == cat) & 
                                 (self.df['Source'] == source)]['Prix_Revente'].median()
                if pd.notna(mediane):
                    self.df.loc[mask, 'Prix_Revente'] = mediane
        
        # Si encore des NaN, utiliser médiane générale par catégorie
        for cat in self.df['Categorie'].unique():
            mask = (self.df['Prix_Revente'].isna()) & (self.df['Categorie'] == cat)
            mediane = self.df[self.df['Categorie'] == cat]['Prix_Revente'].median()
            if pd.notna(mediane):
                self.df.loc[mask, 'Prix_Revente'] = mediane
        
        print(f"✓ Prix_Revente: {prix_missing} remplies par médiane (catégorie + source)")
        
        self.ajouter_rapport('Valeurs manquantes numériques', 
                            'Toutes remplies avec logique conditionnelle')
    
    def traiter_outliers(self):
        """
        ÉTAPE 5: TRAITEMENT DES OUTLIERS
        ==================================
        Correction intelligente selon la nature de chaque variable
        """
        print("\n" + "="*80)
        print("ÉTAPE 5: TRAITEMENT DES OUTLIERS")
        print("="*80)
        
        outliers_corriges = {}
        
        # 5A. POIDS NÉGATIFS - Erreur évidente, prendre valeur absolue
        poids_negatifs = (self.df['Poids'] < 0).sum()
        self.df['Poids'] = self.df['Poids'].abs()
        outliers_corriges['Poids négatifs'] = poids_negatifs
        print(f"✓ {poids_negatifs} poids négatifs corrigés (valeur absolue)")
        
        # 5B. VOLUME NÉGATIF - Erreur, valeur absolue
        vol_negatifs = (self.df['Volume'] < 0).sum()
        self.df['Volume'] = self.df['Volume'].abs()
        outliers_corriges['Volume négatifs'] = vol_negatifs
        print(f"✓ {vol_negatifs} volumes négatifs corrigés (valeur absolue)")
        
        # 5C. POIDS EXTRÊMES - Capping au 99e percentile par catégorie
        for cat in self.df['Categorie'].unique():
            mask_cat = self.df['Categorie'] == cat
            p99 = self.df.loc[mask_cat, 'Poids'].quantile(0.99)
            mask_outlier = mask_cat & (self.df['Poids'] > p99)
            nb_outliers = mask_outlier.sum()
            self.df.loc[mask_outlier, 'Poids'] = p99
            if nb_outliers > 0:
                outliers_corriges[f'Poids extrêmes {cat}'] = nb_outliers
                print(f"✓ {nb_outliers} poids extrêmes corrigés pour {cat} (capping à {p99:.2f})")
        
        # 5D. CONDUCTIVITÉ - Ne doit jamais être > 1 (normalisée)
        cond_sup = (self.df['Conductivite'] > 1).sum()
        self.df.loc[self.df['Conductivite'] > 1, 'Conductivite'] = 1.0
        outliers_corriges['Conductivité > 1'] = cond_sup
        print(f"✓ {cond_sup} conductivités > 1 corrigées (cap à 1.0)")
        
        # 5E. OPACITÉ - Valeurs aberrantes (> 10) remplacées par médiane
        opac_aberrantes = (self.df['Opacite'] > 10).sum()
        for cat in self.df['Categorie'].unique():
            mask_cat = self.df['Categorie'] == cat
            mediane = self.df.loc[mask_cat, 'Opacite'].median()
            mask_aberrant = mask_cat & (self.df['Opacite'] > 10)
            self.df.loc[mask_aberrant, 'Opacite'] = mediane
        outliers_corriges['Opacité aberrante'] = opac_aberrantes
        print(f"✓ {opac_aberrantes} opacités aberrantes (>10) corrigées par médiane")
        
        # 5F. RIGIDITÉ - Doit être entre 1 et 10
        rig_hors_range = ((self.df['Rigidite'] < 1) | (self.df['Rigidite'] > 10)).sum()
        self.df['Rigidite'] = self.df['Rigidite'].clip(1, 10)
        outliers_corriges['Rigidité hors range'] = rig_hors_range
        print(f"✓ {rig_hors_range} rigidités hors range corrigées (clip 1-10)")
        
        # 5G. PRIX_REVENTE NÉGATIFS - Remplacer par 0
        prix_negatifs = (self.df['Prix_Revente'] < 0).sum()
        self.df.loc[self.df['Prix_Revente'] < 0, 'Prix_Revente'] = 0
        outliers_corriges['Prix négatifs'] = prix_negatifs
        print(f"✓ {prix_negatifs} prix négatifs corrigés (mis à 0)")
        
        # 5H. PRIX_REVENTE EXTRÊMES - Capping au 99e percentile
        p99_prix = self.df['Prix_Revente'].quantile(0.99)
        prix_extremes = (self.df['Prix_Revente'] > p99_prix).sum()
        self.df.loc[self.df['Prix_Revente'] > p99_prix, 'Prix_Revente'] = p99_prix
        outliers_corriges['Prix extrêmes'] = prix_extremes
        print(f"✓ {prix_extremes} prix extrêmes corrigés (capping à {p99_prix:.2f})")
        
        self.ajouter_rapport('Correction outliers', 
                            f"Total: {sum(outliers_corriges.values())} valeurs corrigées")
    
    def verifier_coherence(self):
        """
        ÉTAPE 6: VÉRIFICATION DE COHÉRENCE ENTRE COLONNES
        ===================================================
        Corrections basées sur les relations physiques attendues
        """
        print("\n" + "="*80)
        print("ÉTAPE 6: VÉRIFICATION DE COHÉRENCE")
        print("="*80)
        
        # 6A. Densité aberrante (Poids/Volume trop éloigné de la densité typique)
        densites = {
            'Papier': (0.3, 1.5),
            'Plastique': (0.5, 1.5),
            'Verre': (1.5, 3.0),
            'Métal': (2.0, 10.0)
        }
        
        incoherences = 0
        for cat, (densite_min, densite_max) in densites.items():
            mask = (self.df['Categorie'] == cat) & (self.df['Volume'] > 0)
            densite_calculee = self.df.loc[mask, 'Poids'] / self.df.loc[mask, 'Volume']
            
            # Détecter les densités aberrantes
            mask_aberrant = mask & ((densite_calculee < densite_min) | (densite_calculee > densite_max))
            nb_aberrant = mask_aberrant.sum()
            
            if nb_aberrant > 0:
                # Recalculer le volume avec une densité médiane
                densite_mediane = (densite_min + densite_max) / 2
                self.df.loc[mask_aberrant, 'Volume'] = self.df.loc[mask_aberrant, 'Poids'] / densite_mediane
                incoherences += nb_aberrant
                print(f"✓ {nb_aberrant} incohérences densité corrigées pour {cat}")
        
        # 6B. Conductivité incohérente avec catégorie
        # Métal devrait avoir conductivité élevée, autres = 0
        mask_metal_non_cond = (self.df['Categorie'] == 'Métal') & (self.df['Conductivite'] == 0)
        nb_metal = mask_metal_non_cond.sum()
        self.df.loc[mask_metal_non_cond, 'Conductivite'] = \
            self.df[self.df['Categorie'] == 'Métal']['Conductivite'].median()
        
        mask_non_metal_cond = (self.df['Categorie'] != 'Métal') & (self.df['Conductivite'] > 0.1)
        nb_non_metal = mask_non_metal_cond.sum()
        self.df.loc[mask_non_metal_cond, 'Conductivite'] = 0.0
        
        if nb_metal > 0 or nb_non_metal > 0:
            print(f"✓ {nb_metal} métaux avec cond=0 corrigés, {nb_non_metal} non-métaux conducteurs corrigés")
        
        # 6C. Rigidité incohérente avec catégorie
        rigidites_attendues = {
            'Métal': (7, 10),
            'Verre': (8, 10),
            'Plastique': (3, 7),
            'Papier': (1, 5)
        }
        
        for cat, (rig_min, rig_max) in rigidites_attendues.items():
            mask = (self.df['Categorie'] == cat) & \
                   ((self.df['Rigidite'] < rig_min) | (self.df['Rigidite'] > rig_max))
            nb_incoherent = mask.sum()
            if nb_incoherent > 0:
                rig_mediane = (rig_min + rig_max) / 2
                self.df.loc[mask, 'Rigidite'] = rig_mediane
                print(f"✓ {nb_incoherent} rigidités incohérentes corrigées pour {cat}")
        
        self.ajouter_rapport('Vérification cohérence', 
                            f"{incoherences + nb_metal + nb_non_metal} incohérences corrigées")
    
    def creer_features_derivees(self):
        """
        ÉTAPE 7: CRÉATION DE FEATURES DÉRIVÉES
        ========================================
        Nouvelles variables calculées pour enrichir le dataset
        """
        print("\n" + "="*80)
        print("ÉTAPE 7: CRÉATION DE FEATURES DÉRIVÉES")
        print("="*80)
        
        # 7A. Densité calculée
        self.df['Densite'] = self.df['Poids'] / self.df['Volume']
        self.df['Densite'] = self.df['Densite'].replace([np.inf, -np.inf], np.nan)
        self.df['Densite'].fillna(self.df['Densite'].median(), inplace=True)
        print("✓ Densité calculée (Poids/Volume)")
        
        # 7B. Ratio Prix/Poids (rentabilité)
        self.df['Prix_Par_Kg'] = self.df['Prix_Revente'] / self.df['Poids']
        self.df['Prix_Par_Kg'] = self.df['Prix_Par_Kg'].replace([np.inf, -np.inf], np.nan)
        self.df['Prix_Par_Kg'].fillna(0, inplace=True)
        print("✓ Prix par kg calculé (rentabilité)")
        
        # 7C. Score de qualité matériau (combinaison de propriétés)
        # Normaliser les variables
        self.df['Opacite_Norm'] = (self.df['Opacite'] - self.df['Opacite'].min()) / \
                                   (self.df['Opacite'].max() - self.df['Opacite'].min())
        self.df['Rigidite_Norm'] = (self.df['Rigidite'] - 1) / 9  # 1-10 -> 0-1
        
        self.df['Score_Qualite'] = (
            self.df['Opacite_Norm'] * 0.3 + 
            self.df['Rigidite_Norm'] * 0.4 + 
            self.df['Conductivite'] * 0.3
        )
        print("✓ Score de qualité matériau créé")
        
        # 7D. Catégorie de taille (basée sur poids et volume)
        self.df['Categorie_Taille'] = pd.cut(
            self.df['Poids'], 
            bins=[0, 20, 50, 100, np.inf],
            labels=['Petit', 'Moyen', 'Grand', 'Très Grand']
        )
        print("✓ Catégorie de taille créée")
        
        # 7E. Indicateur de rentabilité
        median_prix_kg = self.df['Prix_Par_Kg'].median()
        self.df['Est_Rentable'] = (self.df['Prix_Par_Kg'] > median_prix_kg).astype(int)
        print("✓ Indicateur de rentabilité créé")
        
        # 7F. Encodage de la source et catégorie (pour ML)
        le_cat = LabelEncoder()
        le_source = LabelEncoder()
        
        self.df['Categorie_Encoded'] = le_cat.fit_transform(self.df['Categorie'])
        self.df['Source_Encoded'] = le_source.fit_transform(self.df['Source'])
        print("✓ Encodage des variables catégorielles effectué")
        
        self.ajouter_rapport('Création features', '9 nouvelles features créées')
    
    def normaliser_standardiser(self):
        """
        ÉTAPE 8: NORMALISATION ET STANDARDISATION
        ===========================================
        Prépare les données pour le machine learning
        """
        print("\n" + "="*80)
        print("ÉTAPE 8: NORMALISATION ET STANDARDISATION")
        print("="*80)
        
        # Colonnes à standardiser (moyenne=0, écart-type=1)
        cols_to_standardize = ['Poids', 'Volume', 'Conductivite', 'Opacite', 
                               'Rigidite', 'Prix_Revente', 'Densite', 'Prix_Par_Kg']
        
        scaler = StandardScaler()
        
        for col in cols_to_standardize:
            if col in self.df.columns:
                col_std = f"{col}_Std"
                self.df[col_std] = scaler.fit_transform(self.df[[col]])
        
        print(f"✓ {len(cols_to_standardize)} variables standardisées (suffix _Std)")
        
        self.ajouter_rapport('Normalisation', f"{len(cols_to_standardize)} colonnes standardisées")
    
    def nettoyer_colonnes_temporaires(self):
        """
        ÉTAPE 9: NETTOYAGE DES COLONNES TEMPORAIRES
        =============================================
        Supprime les colonnes intermédiaires utilisées pour l'extraction
        """
        print("\n" + "="*80)
        print("ÉTAPE 9: NETTOYAGE COLONNES TEMPORAIRES")
        print("="*80)
        
        cols_a_supprimer = ['Poids_Extrait', 'Volume_Extrait', 'Rigidite_Extrait',
                           'Opacite_Norm', 'Rigidite_Norm']
        
        cols_supprimees = []
        for col in cols_a_supprimer:
            if col in self.df.columns:
                self.df = self.df.drop(col, axis=1)
                cols_supprimees.append(col)
        
        print(f"✓ {len(cols_supprimees)} colonnes temporaires supprimées")
        self.ajouter_rapport('Nettoyage', f"{len(cols_supprimees)} colonnes temporaires supprimées")
    
    def generer_rapport(self):
        """
        ÉTAPE 10: GÉNÉRATION DU RAPPORT FINAL
        ======================================
        Résumé complet de toutes les modifications
        """
        print("\n" + "="*80)
        print("RAPPORT FINAL DE PRÉTRAITEMENT")
        print("="*80)
        
        print(f"\nDataset original: {self.df_original.shape}")
        print(f"Dataset final: {self.df.shape}")
        print(f"\nModifications effectuées:")
        
        for i, modif in enumerate(self.rapport_modifications, 1):
            print(f"{i}. {modif['Étape']}: {modif['Détails']}")
        
        print(f"\n✓ Valeurs manquantes restantes: {self.df.isnull().sum().sum()}")
        print(f"✓ Nouvelles colonnes créées: {len(self.df.columns) - len(self.df_original.columns)}")
        
        # Statistiques finales
        print("\nStatistiques finales des colonnes numériques:")
        print(self.df.describe())
        
        return pd.DataFrame(self.rapport_modifications)
    
    def executer_pipeline_complet(self):
        """
        PIPELINE COMPLET DE PRÉTRAITEMENT
        ==================================
        Exécute toutes les étapes dans l'ordre optimal
        """
        print("\n" + "🔄"*40)
        print("DÉBUT DU PIPELINE DE PRÉTRAITEMENT COMPLET")
        print("🔄"*40)
        
        # Exécution séquentielle de toutes les étapes
        self.extraire_infos_rapport()
        self.traiter_doublons()
        self.traiter_valeurs_manquantes_categorielles()
        self.traiter_valeurs_manquantes_numeriques()
        self.traiter_outliers()
        self.verifier_coherence()
        self.creer_features_derivees()
        self.normaliser_standardiser()
        self.nettoyer_colonnes_temporaires()
        rapport = self.generer_rapport()
        
        print("\n" + "✅"*40)
        print("PRÉTRAITEMENT TERMINÉ AVEC SUCCÈS!")
        print("✅"*40)
        
        return self.df, rapport


# =============================================================================
# EXÉCUTION DU PIPELINE
# =============================================================================

if __name__ == "__main__":
    # Charger le dataset
    print("Chargement du dataset...")
    df_raw = pd.read_csv('dataset_ProjetML_2026.csv')
    
    # Créer l'instance du préprocesseur
    preprocesseur = PreprocesseurRecyclage(df_raw)
    
    # Exécuter le pipeline complet
    df_clean, rapport = preprocesseur.executer_pipeline_complet()
    
    # Sauvegarder les résultats
    df_clean.to_csv('/home/claude/dataset_pretraite.csv', index=False)
    rapport.to_csv('/home/claude/rapport_pretraitement.csv', index=False)
    
    print(f"\n📁 Fichiers sauvegardés:")
    print(f"  - dataset_pretraite.csv ({df_clean.shape[0]} lignes, {df_clean.shape[1]} colonnes)")
    print(f"  - rapport_pretraitement.csv")
    
    # Afficher un échantillon du dataset nettoyé
    print(f"\n📊 Aperçu du dataset nettoyé:")
    print(df_clean.head())
    print(f"\n📋 Colonnes finales: {list(df_clean.columns)}")

Chargement du dataset...

🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄
DÉBUT DU PIPELINE DE PRÉTRAITEMENT COMPLET
🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄

ÉTAPE 1: EXTRACTION D'INFORMATIONS DU RAPPORT_COLLECTE
✓ Poids extraits du rapport: 9471
✓ Volumes extraits du rapport: 4608
✓ Rigidités extraites du rapport: 6886

ÉTAPE 2: TRAITEMENT DES DOUBLONS
Nombre de doublons détectés: 776
✓ 776 doublons supprimés (gardé les lignes avec plus de données)

ÉTAPE 3: TRAITEMENT VALEURS MANQUANTES - CATÉGORIELLES
✓ Catégorie: 0/511 valeurs remplies par déduction
✓ Source: 0/535 valeurs remplies par déduction
✓ Catégories restantes remplies avec mode: Plastique
✓ Sources restantes remplies avec mode: Collecte_Citoyenne

ÉTAPE 4: TRAITEMENT VALEURS MANQUANTES - NUMÉRIQUES
✓ Poids: 0 du rapport + 930 calculés (densité×volume) + 44 par médiane
✓ Volume: 0 du rapport + 537 calculés (poids/densité) + 0 par médiane
✓ Conductivité: 972 remplies selon catégorie (Métal=médiane, autres=0)
✓ Opacité: 989 remplie

OSError: Cannot save file into a non-existent directory: '\home\claude'